In [ ]:
import os
import sys
import copy

import numpy as np
import pandas as pd
import matplotlib
from matplotlib import pyplot as plt
import logomaker
import colorsys
import matplotlib.colors as mcolors

sys.path.append("/home/akubaney/projects/na_mpnn/evaluation")
from na_eval_utils import read_json_file

In [ ]:
font_path = "./ARIAL.TTF"
matplotlib.font_manager.fontManager.addfont(font_path)

# 2) Tell Matplotlib to load it
plt.rcParams['font.family'] = "Arial"

In [ ]:
# Default DNA color palette for sequence logos
DEFAULT_DNA_COLORS = {
    'A': '#00FF00',
    'C': '#0000FF',
    'G': '#FFA500',
    'T': '#FF0000',
    'U': '#FF0000',
}

# Default style dictionary for sequence logos
DEFAULT_SEQLOGO_STYLE = {
    'figsize': (40 / 25.4, 20 / 25.4),
    'dpi': 300,
    'title_fontsize': None,
    'axis_title_fontsize': None,
    'tick_labelsize': 5,
    'letter_fontsize': 12,
    'letter_color_dict': DEFAULT_DNA_COLORS,
    'show_title': False,
    'title': None,
    'x_label': None, #'Position',
    'y_label': None, #'Information Content (bits)',
    'show_axis_labels': True,
    'show_ticks': True,
    'background_color': 'white',
    'logo_type': 'information',  # 'information' or 'probability'
    'stack_order': 'big_on_top', # 'big_on_top' or 'small_on_top'
    'vpad': 0,  # Vertical padding
    'width': 1,  # Width of each position
    'y_max': 2,  # Maximum y-value for the logo
    'one_index_positions': True,  # Whether to use 1-indexing for x-axis
    'pad_left':  0,   # number of uniform columns to add on the left
    'pad_right': 0,   # number of uniform columns to add on the right
    'x_tick_labels_with_true_seq': True,   # Use A/C/G/T labels when true_sequence is provided
    'axes_rect': [0.175, 0.20, 0.8, 0.75],
    'na_type': 'dna',
    'save_prefix': None,  # Prefix for saving figures
    'save_name': None,
    # New controls for consistent styling with other plots
    'spine_linewidth': 0.7,        # Axis spine thickness
    'tick_width': 0.7,             # Tick width
    'hide_top_right_spines': True, # Toggle for top/right spines
    'desaturate': 0.7,             # 0=no change, 1=fully desaturated
    # Top axis controls
    "show_top_axis": False,
    "top_axis_labels": None,
}

In [ ]:
# The token index, as defined by NA-MPNN.
DNA_RESTYPE_TO_INT = {
    "DA": 21,
    "DC": 22,
    "DG": 23,
    "DT": 24
}
DNA_INT_TO_RESTYPE = {v: k for k, v in DNA_RESTYPE_TO_INT.items()}
DNA_INT_LIST = [
    DNA_RESTYPE_TO_INT["DA"], 
    DNA_RESTYPE_TO_INT["DC"], 
    DNA_RESTYPE_TO_INT["DG"], 
    DNA_RESTYPE_TO_INT["DT"]
]
DNA_INT_TO_REMAPPED_INT = dict(zip(DNA_INT_LIST, range(len(DNA_INT_LIST))))

# Due to shared token mapping.
RNA_INT_TO_RESTYPE = {
    DNA_RESTYPE_TO_INT["DA"]: "A",
    DNA_RESTYPE_TO_INT["DC"]: "C",
    DNA_RESTYPE_TO_INT["DG"]: "G",
    DNA_RESTYPE_TO_INT["DT"]: "U",
}

In [ ]:
def load_predicted_pwm_and_true_sequence(
    json_path, num_chains_to_plot, na_type = "dna"
):
    """
    Load predicted PWM and true sequence data from JSON files.
    Args:
        json_path (str): Path to the JSON file containing evaluation results
        num_chains_to_plot (int): Number of DNA chains to include in the plot
        na_type (str): Type of nucleic acid ('dna' or 'rna')
    Returns:
        tuple: (reference_aligned_ppm, subject_predicted_ppm, 
            subject_true_sequence)
    """
    # Load the score and prediction JSON files.
    score_json_dict = read_json_file(json_path)
    prediction_json_dict = read_json_file(score_json_dict["subject_path"])

    # Extract the reference aligned PPM and mask.
    reference_aligned_ppm = np.array(
        score_json_dict["aligned_ppm"], dtype = np.float64
    )
    reference_ppm_mask = np.array(
        score_json_dict["ppm_mask"], dtype = np.int32
    )

    # Extract the subject predicted PPM, mask, and NA mask.
    subject_predicted_ppm = np.array(
        prediction_json_dict["predicted_ppm_na_mpnn_format"], 
        dtype = np.float64
    )
    subject_mask = np.array(
        prediction_json_dict["mask"], dtype = np.int32
    )
    subject_na_mask = np.array(
        prediction_json_dict[f"{na_type}_mask"],
        dtype = np.int32
    )
    subject_true_sequence = np.array(
        prediction_json_dict["true_sequence_na_mpnn_format"],
        dtype = np.int32
    )

    # Compute the position mask, which indicates which positions have both a
    # reference (ground truth) PPM and a subject (predicted) PPM, and are also
    # part of the NA sequence.
    position_mask = (reference_ppm_mask == 1) & (subject_mask == 1) & \
        (subject_na_mask == 1)
    
    # Get the chain labels of the NA, and plot the first `num_chains_to_plot` 
    # unique chains.
    subject_chain_labels = np.array(
        prediction_json_dict["chain_labels"], 
        dtype = np.int32
    )
    na_chain_labels = subject_chain_labels[position_mask]

    if num_chains_to_plot is None:
        chains_to_plot = np.unique(na_chain_labels)
    else:
        chains_to_plot = np.unique(na_chain_labels)[:num_chains_to_plot]

    # Compute a mask for the positions that are in the chains we want to plot.
    chain_mask = np.isin(subject_chain_labels, chains_to_plot)

    # Combine the position mask with the chain mask.
    position_mask = np.logical_and(position_mask, chain_mask)
    
    # Subset the positions.
    reference_aligned_ppm = reference_aligned_ppm[position_mask]
    subject_predicted_ppm = subject_predicted_ppm[position_mask]
    subject_true_sequence = subject_true_sequence[position_mask]

    # Subset the residue types.
    reference_aligned_ppm = reference_aligned_ppm[:, DNA_INT_LIST]
    subject_predicted_ppm = subject_predicted_ppm[:, DNA_INT_LIST]

    return reference_aligned_ppm, subject_predicted_ppm, subject_true_sequence

In [ ]:
def plot_seq_logo(ppm, style = None, true_sequence=None):
    """
    Create a sequence logo from a position probability matrix (PPM).
    Args:
        ppm (numpy.ndarray): Position probability matrix with shape 
            (positions, 4)
        style (dict, optional): Style dictionary to control appearance. 
            See DEFAULT_SEQLOGO_STYLE.
        true_sequence (array-like, optional): Sequence in NA-MPNN int encoding
            aligned to ppm rows; used for x-axis labels as A/C/G/T when enabled.
    Returns:
        matplotlib.figure.Figure: The generated sequence logo figure
    """
    merged_style = copy.deepcopy(DEFAULT_SEQLOGO_STYLE)
    if style is not None:
        merged_style.update(style)

    # Optional uniform padding on either side
    pad_left  = int(merged_style.get('pad_left', 0))
    pad_right = int(merged_style.get('pad_right', 0))
    if pad_left > 0 or pad_right > 0:
        num_bases = ppm.shape[1]
        # left pad ppm
        if pad_left > 0:
            left_block = np.full((pad_left, num_bases), 1.0 / num_bases)
            ppm = np.vstack((left_block, ppm))
        # right pad ppm
        if pad_right > 0:
            right_block = np.full((pad_right, num_bases), 1.0 / num_bases)
            ppm = np.vstack((ppm, right_block))
        
        # pad true sequence; only necessary if the na-mpnn true sequence needs
        # to be padded.
        if true_sequence is not None and (len(true_sequence) != len(ppm)):
            true_sequence = np.concatenate((
                np.full(pad_left, np.nan, dtype=true_sequence.dtype),
                true_sequence,
                np.full(pad_right, np.nan, dtype=true_sequence.dtype)
            ))

    # Normalize the PPM; necessary due to floating point errors.
    ppm = np.array(ppm)
    ppm = ppm / np.sum(ppm, axis = -1, keepdims = True)

    # Create the DataFrame for logomaker.
    na_names = (
        ['A', 'C', 'G', 'T']
        if merged_style.get('na_type') == 'dna'
        else ['A', 'C', 'G', 'U']
    )
    df = pd.DataFrame(ppm, columns = na_names)
    df = logomaker.transform_matrix(
        df,
        from_type = 'probability',
        to_type = 'information',
        normalize_values = False
    )

    # Desaturation helper: reduces saturation in HLS space by factor (0..1)
    def _desaturate_hex(hex_color: str, factor: float) -> str:
        try:
            rgb = mcolors.to_rgb(hex_color)
        except Exception:
            return hex_color
        h, l, s = colorsys.rgb_to_hls(*rgb)
        s2 = max(0.0, s * (1.0 - float(factor)))
        r2, g2, b2 = colorsys.hls_to_rgb(h, l, s2)
        return mcolors.to_hex((r2, g2, b2))

    desat_factor = float(merged_style.get('desaturate', 0.0) or 0.0)
    base_colors = merged_style.get('letter_color_dict', DEFAULT_DNA_COLORS) or DEFAULT_DNA_COLORS
    if desat_factor and desat_factor > 1e-12:
        letter_color_dict = {k: _desaturate_hex(v, desat_factor) for k, v in base_colors.items()}
    else:
        letter_color_dict = base_colors

    # Create the figure and axes for the logo.
    fig = plt.figure(
        figsize = merged_style['figsize'], dpi = merged_style['dpi'], constrained_layout = True
    )
    ax = fig.add_axes(merged_style['axes_rect'])

    # Plot the sequence logo.
    logo = logomaker.Logo(
        df,
        ax = ax,
        color_scheme = letter_color_dict,
        stack_order = merged_style['stack_order'],
        vpad = merged_style['vpad'],
        width = merged_style['width']
    )
    
    # Style the letters in the logo.
    logo.style_glyphs(
        color_scheme = letter_color_dict,
        fontsize = merged_style['letter_fontsize']
    )

    # Set the y-axis max.
    if merged_style['y_max'] is not None:
        ax.set_ylim(0, merged_style['y_max'])

    # Show title.
    if merged_style['show_title'] and merged_style['title']:
        ax.set_title(
            merged_style['title'], fontsize = merged_style['title_fontsize']
        )
    
    # Axes labels and ticks.
    if merged_style['show_axis_labels']:
        if merged_style['x_label']:
            ax.set_xlabel(
                merged_style['x_label'], 
                fontsize = merged_style['axis_title_fontsize']
            )
        if merged_style['y_label']:
            ax.set_ylabel(
                merged_style['y_label'], 
                fontsize = merged_style['axis_title_fontsize']
            )
    else:
        ax.set_xlabel("")
        ax.set_ylabel("")

    if not merged_style['show_ticks']:
        ax.set_xticks([])
        ax.set_yticks([])
    else:
        # X tick labels: numeric or true sequence letters
        num_pos = df.shape[0]
        ax.set_xticks(np.arange(num_pos))
        use_true_seq = bool(merged_style.get('x_tick_labels_with_true_seq', False)) and (true_sequence is not None)
        if use_true_seq:
            int_to_restype = (
                DNA_INT_TO_RESTYPE
                if merged_style.get('na_type') == 'dna'
                else RNA_INT_TO_RESTYPE
            )
            labels = [int_to_restype.get(res_int, " ")[-1] for res_int in true_sequence]
            ax.set_xticklabels(labels, fontsize = merged_style['tick_labelsize'])
        else:
            # Optional 1-indexing of the x-axis.
            if merged_style.get('one_index_positions', True):
                ax.set_xticklabels(
                    np.arange(1, num_pos + 1),
                    fontsize = merged_style['tick_labelsize']
                )
            else:
                ax.set_xticklabels(
                    np.arange(num_pos),
                    fontsize = merged_style['tick_labelsize']
                )
        
        ax.tick_params(
            axis = "both", labelsize = merged_style['tick_labelsize']
        )
    
    # Set the background color.
    ax.set_facecolor(merged_style['background_color'])
    fig.patch.set_facecolor(merged_style['background_color'])

    # Apply spine/tick width and optionally hide top/right spines
    spine_lw = merged_style.get('spine_linewidth', None)
    tick_w = merged_style.get('tick_width', None)
    hide_tr = merged_style.get('hide_top_right_spines', False)

    if spine_lw is not None:
        for spine in ax.spines.values():
            spine.set_linewidth(spine_lw)
    if tick_w is not None:
        ax.tick_params(width=tick_w)
    if hide_tr:
        if 'top' in ax.spines:
            ax.spines['top'].set_visible(False)
        if 'right' in ax.spines:
            ax.spines['right'].set_visible(False)

    # Optional top axis with user-specified labels.                                                                                                                                           
    if merged_style.get('show_top_axis', False) and \
            merged_style.get('top_axis_labels') is not None:                                                                                                                                  
        top_labels = merged_style['top_axis_labels']
        num_pos = df.shape[0]                                                                                                                                                                 
        ax_top = ax.twiny()
        ax_top.set_xlim(ax.get_xlim())                                                                                                                                                        
        ax_top.set_xticks(np.arange(num_pos))
        ax_top.set_xticklabels(                                                                                                                                                               
            top_labels, fontsize = merged_style['tick_labelsize']
        )                                                                                                                                                                                     
        ax_top.tick_params(
            axis = 'x', labelsize = merged_style['tick_labelsize']                                                                                                                            
        )       
        if tick_w is not None:
            ax_top.tick_params(width = tick_w)
        if spine_lw is not None:                                                                                                                                                              
            for spine in ax_top.spines.values():
                spine.set_linewidth(spine_lw)                                                                                                                                                 
        # Keep only the top spine on the twin to avoid overlaps with ax.
        ax_top.spines['right'].set_visible(False)                                                                                                                                             
        ax_top.spines['left'].set_visible(False)
        ax_top.spines['bottom'].set_visible(False)

    # Save the figure if a save name is provided.
    if merged_style['save_name']:
        plt.savefig(
            merged_style['save_name'], 
            dpi = merged_style['dpi'],
            pad_inches = 0
        )
    
    return fig

def plot_seq_logo_comparison(
    id, 
    na_mpnn_num_chains_to_plot = 1, 
    deeppbs_num_chains_to_plot = 1, 
    style = None, 
    show_all_titles = False,
    na_mpnn_pad_left=0,
    na_mpnn_pad_right=0,
    deeppbs_pad_left=0,
    deeppbs_pad_right=0
):
    """
    Create a comparison of sequence logos between reference, NA-MPNN 
    prediction, and DeepPBS prediction.
    
    Args:
        id (str): Identifier for the structure to compare
        na_mpnn_num_chains_to_plot (int): Number of DNA chains to plot for 
            NA-MPNN.
        deeppbs_num_chains_to_plot (int): Number of DNA chains to plot for 
            DeepPBS.
        style (dict, optional): Style dictionary for the sequence logos.
        show_all_titles (bool, optional): Whether to show all titles in the
            sequence logos.
        na_mpnn_pad_left (int, optional): Number of uniform columns to add
            on the left for NA-MPNN.
        na_mpnn_pad_right (int, optional): Number of uniform columns to add
            on the right for NA-MPNN.
        deeppbs_pad_left (int, optional): Number of uniform columns to add 
            on the left for DeepPBS.
        deeppbs_pad_right (int, optional): Number of uniform columns to add
            on the right for DeepPBS.
    Returns:
        list: List of matplotlib figures for the three sequence logos
    """
    # Overall output directory for the evaluation outputs.
    base_folder = os.path.join(
        "/",
        "home",
        "akubaney",
        "projects",
        "na_mpnn",
        "evaluation",
        "evaluation_outputs",
        "specificity_test_scores",
    )
    
    # Load the reference and predicted PPMs for NA-MPNN and DeepPBS.
    na_mpnn_score_path = os.path.join(
        base_folder, "na_mpnn", id, f"{id}.json"
    )
    deeppbs_score_path = os.path.join(
        base_folder, "deeppbs", id, f"{id}.json"
    )
    ref_ppm_na_mpnn, pred_ppm_na_mpnn, true_seq_na_mpnn = \
        load_predicted_pwm_and_true_sequence(
            na_mpnn_score_path, na_mpnn_num_chains_to_plot
    )
    ref_ppm_deeppbs, pred_ppm_deeppbs, true_seq_deeppbs = \
        load_predicted_pwm_and_true_sequence(
            deeppbs_score_path, deeppbs_num_chains_to_plot
    )

    print(f"NA-MPNN true sequence: {list(map(lambda res_int: DNA_INT_TO_RESTYPE[res_int], true_seq_na_mpnn))}")
    print(f"DeepPBS true sequence: {list(map(lambda res_int: DNA_INT_TO_RESTYPE[res_int], true_seq_deeppbs))}")
    
    save_prefix = style.get('save_prefix', None) if style else None

    figures = []
    # Plot the aligned reference NA-MPNN PPM.
    ref_na_mpnn_style = copy.deepcopy(style) if style else {}
    ref_na_mpnn_style.update({
        'show_title': show_all_titles,
        'title': f"NA-MPNN Reference - {id}",
        'pad_left': na_mpnn_pad_left,
        'pad_right': na_mpnn_pad_right,
        'save_name': f"{save_prefix}_{id}_na_mpnn_reference.svg" if save_prefix else None
    })
    print(ref_na_mpnn_style["title"])
    figures.append(plot_seq_logo(ref_ppm_na_mpnn, ref_na_mpnn_style, true_sequence=true_seq_na_mpnn))

    # Plot the aligned reference DeepPBS PPM.
    ref_deeppbs_style = copy.deepcopy(style) if style else {}
    ref_deeppbs_style.update({
        'show_title': show_all_titles,
        'title': f"DeepPBS Reference - {id}",
        'pad_left': deeppbs_pad_left,
        'pad_right': deeppbs_pad_right,
        'save_name': 
            f"{save_prefix}_{id}_deep_pbs_reference.svg" if save_prefix else None
    })
    print(ref_deeppbs_style["title"])
    figures.append(plot_seq_logo(ref_ppm_deeppbs, ref_deeppbs_style, true_sequence=true_seq_na_mpnn))

    # Plot the NA-MPNN predicted PPM.
    na_mpnn_style = copy.deepcopy(style) if style else {}
    na_mpnn_style.update({
        'show_title': show_all_titles,
        'title': f"NA-MPNN Prediction - {id}",
        'pad_left': na_mpnn_pad_left,
        'pad_right': na_mpnn_pad_right,
        'save_name': f"{save_prefix}_{id}_na_mpnn_prediction.svg" if save_prefix else None
    })
    print(na_mpnn_style["title"])
    figures.append(plot_seq_logo(pred_ppm_na_mpnn, na_mpnn_style, true_sequence=true_seq_na_mpnn))
    
    # Plot the DeepPBS predicted PPM.
    deeppbs_style = copy.deepcopy(style) if style else {}
    deeppbs_style.update({
        'show_title': show_all_titles,
        'title': f"DeepPBS Prediction - {id}",
        'pad_left': deeppbs_pad_left,
        'pad_right': deeppbs_pad_right,
        'save_name': f"{save_prefix}_{id}_deep_pbs_prediction.svg" if save_prefix else None
    })
    print(deeppbs_style["title"])
    figures.append(plot_seq_logo(pred_ppm_deeppbs, deeppbs_style, true_sequence=true_seq_na_mpnn))

    return figures

# Fig 4a (Data augmentation demonstration)

In [ ]:
id = "DDB_G0278225_AAATGCCA"

base_folder = os.path.join(
    "/",
    "home",
    "akubaney",
    "projects",
    "na_mpnn",
    "evaluation",
    "evaluation_outputs",
    "specificity_test_scores",
)

# Load the reference and predicted PPMs for NA-MPNN and DeepPBS.
na_mpnn_score_path = os.path.join(
    base_folder, "na_mpnn", id, f"{id}.json"
)
ref_ppm_na_mpnn, pred_ppm_na_mpnn, true_seq_na_mpnn = \
    load_predicted_pwm_and_true_sequence(
        na_mpnn_score_path, 1
)

In [ ]:
list(map(lambda res_int: DNA_INT_TO_RESTYPE[res_int], true_seq_na_mpnn))

In [ ]:
ref_ppm_na_mpnn_df = pd.DataFrame(
    ref_ppm_na_mpnn, 
    columns = ["DA", "DC", "DG", "DT"]
) 
ref_ppm_na_mpnn_df

In [ ]:
# Data Augmentation Demonstration
fig = plot_seq_logo(ref_ppm_na_mpnn, true_sequence=true_seq_na_mpnn, style = {
    "x_tick_labels_with_true_seq": False,
    "save_name": "/home/akubaney/projects/na_mpnn/figures/matplotlib/data_augmentation_demo.svg",
    'figsize': (35 / 25.4, 35 / 25.4),
    'axes_rect': [0.175, 0.15, 0.8, 0.825],
})

In [ ]:
fig = plot_seq_logo(ref_ppm_na_mpnn, true_sequence=true_seq_na_mpnn, style = {
    "x_tick_labels_with_true_seq": True,
    "save_name": "/home/akubaney/projects/na_mpnn/figures/matplotlib/na_mpnn_to_ppm.svg",
    'figsize': (32 / 25.4, 20 / 25.4),
})

# Fig 4b (Best Distillation)

In [ ]:
plot_seq_logo_comparison(
    "SCHCODRAFT_80572_AAAGCCAC",
    style = {
        "save_prefix": "/home/akubaney/projects/na_mpnn/figures/matplotlib/best_from_distillation"
    }
)

In [ ]:
plot_seq_logo_comparison(
    "NCU09387_AAACAAAG",
    style = {
        "save_prefix": "/home/akubaney/projects/na_mpnn/figures/matplotlib/best_from_distillation"
    },
    deeppbs_pad_right = 2
)

In [ ]:
plot_seq_logo_comparison(
    "DDB_G0278225_AAATGCCA",
    style = {
        "save_prefix": "/home/akubaney/projects/na_mpnn/figures/matplotlib/best_from_distillation"
    }
)

# Fig 6 (Same Protein)

In [ ]:
plot_seq_logo_comparison(
    "SCHCODRAFT_80572_AAAGCCAC",
    na_mpnn_pad_left = 1,
    deeppbs_pad_left = 1,
    style = {
        "save_prefix": "/home/akubaney/projects/na_mpnn/figures/matplotlib/same_protein"
    }
)

In [ ]:
plot_seq_logo_comparison(
    "SCHCODRAFT_80572_AACGCCAT",
    na_mpnn_pad_right = 1,
    deeppbs_pad_right = 1,
    style = {
        "save_prefix": "/home/akubaney/projects/na_mpnn/figures/matplotlib/same_protein"
    }
)

In [ ]:
plot_seq_logo_comparison(
    "SCHCODRAFT_80572_AACGCCAC",
    style = {
        "save_prefix": "/home/akubaney/projects/na_mpnn/figures/matplotlib/same_protein"
    }
)

# Fig 7a (Best Crystal)

In [ ]:
plot_seq_logo_comparison(
    "1am9", 
    na_mpnn_num_chains_to_plot=2, 
    deeppbs_num_chains_to_plot=1,
    style = {
        "save_prefix": "/home/akubaney/projects/na_mpnn/figures/matplotlib/best_from_crystal",
        "figsize": (80 / 25.4, 20 / 25.4),
        "axes_rect": [0.1, 0.20, 0.875, 0.75]
    }
)

In [ ]:
plot_seq_logo_comparison(
    "6u81",
    style = {
        "save_prefix": "/home/akubaney/projects/na_mpnn/figures/matplotlib/best_from_crystal",
        "figsize": (55 / 25.4, 20 / 25.4),
        "axes_rect": [0.1, 0.20, 0.875, 0.75]
    },
    deeppbs_pad_left = 4
)

# RNA

In [ ]:
id = "1b7f"

rna_json_path = os.path.join(
    "/home/akubaney/projects/na_mpnn/evaluation/evaluation_outputs",
    "specificity_test_rna",
    "na_mpnn",
    id,
    "specificity_json",
    f"{id}.json",
)

rna_output = read_json_file(rna_json_path)

mask = np.array(rna_output["mask"], dtype = np.int32)
rna_mask = np.array(rna_output["rna_mask"], dtype = np.int32)
rna_chain_labels = np.array(rna_output["chain_labels"], dtype = np.int32)
chains_to_plot = np.unique(rna_chain_labels[rna_mask == 1])[:1]
position_mask = (mask == 1) & (rna_mask == 1) & \
    np.isin(rna_chain_labels, chains_to_plot)

rna_ppm = np.array(
    rna_output["predicted_ppm_na_mpnn_format"],
    dtype = np.float64,
)[position_mask][:, DNA_INT_LIST]
rna_true_sequence = np.array(
    rna_output["true_sequence_na_mpnn_format"],
    dtype = np.int32,
)[position_mask]

list(map(lambda res_int: RNA_INT_TO_RESTYPE[res_int], rna_true_sequence))

fig = plot_seq_logo(
    rna_ppm,
    true_sequence = rna_true_sequence,
    style = {
        "na_type": "rna",
        "save_name": "/home/akubaney/projects/na_mpnn/figures/matplotlib/rna_1b7f_prediction.svg",
        "figsize": (40 / 25.4, 40 / 25.4),
        "show_top_axis": True,
        "top_axis_labels": ["-", "-", "U", "G", "U", "U", "U", "U", "U", "U", "U", "-"],
        "axes_rect": [0.175, 0.20, 0.8, 0.65]
    },
)

In [ ]:
id = "9imb"

rna_json_path = os.path.join(
    "/home/akubaney/projects/na_mpnn/evaluation/evaluation_outputs",
    "specificity_test_rna",
    "na_mpnn",
    id,
    "specificity_json",
    f"{id}.json",
)

rna_output = read_json_file(rna_json_path)

mask = np.array(rna_output["mask"], dtype = np.int32)
rna_mask = np.array(rna_output["rna_mask"], dtype = np.int32)
rna_chain_labels = np.array(rna_output["chain_labels"], dtype = np.int32)
chains_to_plot = np.unique(rna_chain_labels[rna_mask == 1])[:1]

# mask to interesting region.
interesting_mask = np.zeros_like(mask, dtype = bool)
interesting_mask[466:474] = True

position_mask = (mask == 1) & (rna_mask == 1) & \
    np.isin(rna_chain_labels, chains_to_plot) & interesting_mask

rna_ppm = np.array(
    rna_output["predicted_ppm_na_mpnn_format"],
    dtype = np.float64,
)[position_mask][:, DNA_INT_LIST]
rna_true_sequence = np.array(
    rna_output["true_sequence_na_mpnn_format"],
    dtype = np.int32,
)[position_mask]

list(map(lambda res_int: RNA_INT_TO_RESTYPE[res_int], rna_true_sequence))

fig = plot_seq_logo(
    rna_ppm,
    true_sequence = rna_true_sequence,
    style = {
        "na_type": "rna",
        "figsize": (40 / 25.4, 40 / 25.4),
        "save_name": "/home/akubaney/projects/na_mpnn/figures/matplotlib/rna_9imb_prediction.svg",
        "show_top_axis": True,
        "top_axis_labels": ["-", "-", "C", "U", "C", "C/U", "A", "-"],
        "axes_rect": [0.175, 0.20, 0.8, 0.65],
    },
)


# 1GT0 PPM Alignment Schematic

## Setup

In [ ]:
PPM_ALIGNMENT_OUT_DIR = (
    "/home/akubaney/projects/na_mpnn/figures/matplotlib/"
    "ppm_alignment_1gt0"
)
os.makedirs(PPM_ALIGNMENT_OUT_DIR, exist_ok = True)

PPM_ROOT = (
    "/home/akubaney/projects/na_mpnn/data/datasets/rcsb_cif_na/"
    "preprocessed_ppms"
)

SEQUENCES_PATH = (
    "/home/akubaney/projects/na_mpnn/data/datasets/rcsb_cif_na/"
    "preprocessed_data/sequences/1gt0.csv"
)
sequence_df = pd.read_csv(SEQUENCES_PATH)
DNA_CHAIN_SEQUENCES = dict(
    zip(
        sequence_df.loc[
            sequence_df["chain_type"] == "polydeoxyribonucleotide",
            "chain_id",
        ],
        sequence_df.loc[
            sequence_df["chain_type"] == "polydeoxyribonucleotide",
            "sequence",
        ],
    )
)

CHAIN_A_SEQUENCE = DNA_CHAIN_SEQUENCES["A"]
CHAIN_B_SEQUENCE = DNA_CHAIN_SEQUENCES["B"]

PO2F1_GROUP = [
    ("PO2F1_HUMAN.H11MO.0.C", os.path.join(PPM_ROOT, "PO2F1_HUMAN.H11MO.0.C.csv")),
    ("MA1962.1.jaspar", os.path.join(PPM_ROOT, "MA1962.1.jaspar.csv")),
    ("MA0785.1.jaspar", os.path.join(PPM_ROOT, "MA0785.1.jaspar.csv")),
]

SOX2_GROUP = [
    ("SOX2_MOUSE.H11MO.0.A", os.path.join(PPM_ROOT, "SOX2_MOUSE.H11MO.0.A.csv")),
    ("MA0143.3.jaspar", os.path.join(PPM_ROOT, "MA0143.3.jaspar.csv")),
]

PO2F1_SELECTED_NAME, PO2F1_SELECTED_PATH = PO2F1_GROUP[0]
SOX2_SELECTED_NAME, SOX2_SELECTED_PATH = SOX2_GROUP[0]

DNA_BASE_TO_TOKEN = {
    "A": DNA_RESTYPE_TO_INT["DA"],
    "C": DNA_RESTYPE_TO_INT["DC"],
    "G": DNA_RESTYPE_TO_INT["DG"],
    "T": DNA_RESTYPE_TO_INT["DT"],
}

def load_dna_ppm(ppm_path):
    ppm_df = pd.read_csv(ppm_path)
    return ppm_df[["A", "C", "G", "T"]].to_numpy(dtype = np.float64)

def reverse_complement_ppm(ppm):
    # PPM columns are A, C, G, T. Reverse rows, then swap A/T and C/G.
    return np.flip(np.flip(ppm, axis = 1), axis = 0).copy()

def dna_true_sequence(sequence):
    return np.array([DNA_BASE_TO_TOKEN[base] for base in sequence], dtype = np.int32)

def plot_schematic_ppm_logo(
    ppm,
    save_stem,
    title = None,
    true_sequence = None,
    bottom_axis_labels = None,
    show_bottom_axis_labels = True,
    show_y_axis = True,
    width_mm = None,
    height_mm = 20,
):
    """
    Plot a schematic PPM/logo with one bottom x-axis only.

    Bottom-axis choices:
    - bottom_axis_labels=range(...): show chain positions
    - true_sequence=dna_true_sequence(...): show crystal bases
    - neither: show motif/logo positions 1..L
    - show_bottom_axis_labels=False: show no bottom x-axis labels

    Do not combine these bottom-axis choices for the same plot.
    """
    if bottom_axis_labels is not None and true_sequence is not None:
        raise ValueError("Choose either bottom_axis_labels or true_sequence, not both.")
    if not show_bottom_axis_labels and (
        bottom_axis_labels is not None or true_sequence is not None
    ):
        raise ValueError("Use show_bottom_axis_labels=False without bottom-axis labels.")

    if width_mm is None:
        width_mm = 2.8 * ppm.shape[0]

    axes_rect = [0.13, 0.20, 0.84, 0.74]
    if not show_y_axis:
        axes_rect[0] = 0.03
        axes_rect[2] = 0.94
    if show_y_axis:
        axes_rect[3] = 0.65

    if not show_bottom_axis_labels:
        axes_rect[1] = 0.06
        axes_rect[3] += 0.14

    if title is None:
        axes_rect[3] += 0.08

    save_name = None if save_stem is None else os.path.join(
        PPM_ALIGNMENT_OUT_DIR, f"{save_stem}.svg"
    )

    style = {
        "save_name": save_name,
        "figsize": (width_mm / 25.4, height_mm / 25.4),
        "dpi": 300,
        "axes_rect": axes_rect,
        "show_title": title is not None,
        "title": title,
        "title_fontsize": 5,
        "tick_labelsize": 5,
        "letter_fontsize": 12,
        "x_tick_labels_with_true_seq": true_sequence is not None,
        "desaturate": 0.7,
    }

    fig = plot_seq_logo(ppm, true_sequence = true_sequence, style = style)

    if bottom_axis_labels is not None:
        ax = fig.axes[0]
        ax.set_xticks(np.arange(ppm.shape[0]))
        ax.set_xticklabels(
            [str(label) for label in bottom_axis_labels],
            fontsize = style["tick_labelsize"],
        )

    elif not show_bottom_axis_labels:
        ax = fig.axes[0]
        ax.set_xticks([])
    
    if not show_y_axis:
        ax = fig.axes[0]
        ax.set_yticks([])
        ax.set_ylabel("")
        ax.spines["left"].set_visible(False)

    if save_name is not None:
        fig.savefig(save_name, dpi = style["dpi"], pad_inches = 0)


    return fig

PPM selection

In [ ]:
for group_label, ppm_group in [
    ("group0_po2f1", PO2F1_GROUP),
    ("group1_sox2", SOX2_GROUP),
]:
    for ppm_name, ppm_path in ppm_group:
        ppm = load_dna_ppm(ppm_path)
        print(ppm_name)
        plot_schematic_ppm_logo(
            ppm,
            save_stem = f"panel_a_{group_label}_{ppm_name}",
            show_bottom_axis_labels=False,
            show_y_axis=False,
            height_mm=23,
        )

Reverse complement

In [ ]:
sox2_forward = load_dna_ppm(SOX2_SELECTED_PATH)
sox2_revcomp = reverse_complement_ppm(sox2_forward)

plot_schematic_ppm_logo(
    sox2_forward,
    save_stem = "panel_b_sox2_forward",
    show_bottom_axis_labels = False,
    show_y_axis=False,
    height_mm=23,
)

plot_schematic_ppm_logo(
    sox2_revcomp,
    save_stem = "panel_b_sox2_reverse_complement",
    show_bottom_axis_labels = False,
    show_y_axis=False,
    height_mm=23,
)

"Done!"

Align against all chains

In [ ]:
plot_schematic_ppm_logo(
    sox2_revcomp,
    save_stem = "panel_c_sox2_reverse_chain_search_candidate",
    show_bottom_axis_labels = False,
    show_y_axis=False,
    height_mm=23,
)

"Done!"

Align against one chain

In [ ]:
sox2_forward = load_dna_ppm(SOX2_SELECTED_PATH)
sox2_revcomp = reverse_complement_ppm(sox2_forward)

chain_a_sox2_start = 8
chain_a_sox2_end = 22
sox2_ppm_start = 1
sox2_ppm_end = 15

sox2_chain_a_ppm = sox2_revcomp[sox2_ppm_start - 1 : sox2_ppm_end]
sox2_chain_a_sequence = CHAIN_A_SEQUENCE[chain_a_sox2_start - 1 : chain_a_sox2_end]

In [ ]:
print(f"Chain A ({len(CHAIN_A_SEQUENCE)} nt): {CHAIN_A_SEQUENCE}")

In [ ]:
plot_schematic_ppm_logo(
    sox2_chain_a_ppm,
    save_stem = "panel_d_sox2_revcomp_chain_a_8_22",
    show_bottom_axis_labels = False,
    show_y_axis=False,
    height_mm=23,
)
"Done!"

In [ ]:
plot_schematic_ppm_logo(
    sox2_chain_a_ppm,
    save_stem = "panel_d_sox2_revcomp_chain_a_8_22_with_true_sequence",
    true_sequence = dna_true_sequence(sox2_chain_a_sequence),
    height_mm=23,
)
"Done!"

Conflict resolution

In [ ]:
po2f1_forward = load_dna_ppm(PO2F1_SELECTED_PATH)
sox2_revcomp = reverse_complement_ppm(load_dna_ppm(SOX2_SELECTED_PATH))

# Chain A positions 8-22.
chain_a_panel_d_sequence = CHAIN_A_SEQUENCE[7:22]

# Show the two selected aligned motifs as their selected PPM chunks.
po2f1_panel_d = po2f1_forward[2:14]  # chain A 9-20, PPM 3-14
sox2_panel_d = sox2_revcomp[:15]     # chain A 8-22, PPM 1-15

# Keep the final merged motif on chain A positions 8-22.
final_panel_d = sox2_panel_d.copy()
final_panel_d[1] = po2f1_forward[2]         # chain A 9
final_panel_d[5:9] = po2f1_forward[6:10]    # chain A 13-16

for ppm, save_stem, sequence in [
    (po2f1_panel_d, "panel_e_po2f1_selected_chain_a_9_20", CHAIN_A_SEQUENCE[8:20]),
    (sox2_panel_d, "panel_e_sox2_revcomp_chain_a_8_22", chain_a_panel_d_sequence),
]:
    plot_schematic_ppm_logo(
        ppm,
        save_stem = save_stem,
        show_bottom_axis_labels = False,
        show_y_axis=False,
        height_mm=23,
    )
    plot_schematic_ppm_logo(
        ppm,
        save_stem = None,
        true_sequence = dna_true_sequence(sequence),
        show_y_axis=False,
    )

fig = plot_schematic_ppm_logo(
    final_panel_d,
    save_stem = "panel_e_final",
    true_sequence = dna_true_sequence(chain_a_panel_d_sequence),
    height_mm=23,
)
plt.show()
